# 自编码器 (Autoencoder)

## 核心思想

自编码器是一种**无监督学习**模型, 目标是学习数据的压缩表示:

```
输入 x -> 编码器 -> 潜在表示 z -> 解码器 -> 重建 x_hat
         (压缩)    (低维)     (还原)

目标: x_hat 尽可能接近 x
```

为什么有用? 强制通过低维瓶颈, 网络必须学到数据中最重要的特征。

## 本课内容
1. 基础自编码器
2. 去噪自编码器 (DAE)
3. 变分自编码器 (VAE)
4. 稀疏自编码器
5. 自编码器在RFFI中的应用
6. 对比学习简述

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
torch.manual_seed(42)

## 1. 基础自编码器

### 结构

```
编码器: x (高维) -> z (低维)
解码器: z (低维) -> x_hat (高维)

损失: L = ||x - x_hat||^2  (MSE重建损失)
```

关键: 瓶颈层维度 < 输入维度, 强制压缩。

In [ ]:
# 生成模拟数据
n_samples = 2000
n_features = 20
n_classes = 5

X = []
y = []
for c in range(n_classes):
    center = np.random.randn(n_features) * 3
    samples = center + np.random.randn(n_samples // n_classes, n_features) * 0.5
    X.append(samples)
    y.extend([c] * (n_samples // n_classes))

X = np.concatenate(X, axis=0).astype(np.float32)
y = np.array(y)
X = (X - X.mean()) / X.std()

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

print(f"数据: {X.shape}, 类别数: {n_classes}")

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim=20, latent_dim=3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
        )
    
    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

model_ae = Autoencoder(input_dim=20, latent_dim=3)
print(f"参数量: {sum(p.numel() for p in model_ae.parameters()):,}")
print(f"压缩比: 20维 -> 3维 (6.7x)")

In [ ]:
# 训练
optimizer = optim.Adam(model_ae.parameters(), lr=1e-3)
criterion = nn.MSELoss()

losses = []
for epoch in range(100):
    epoch_loss = 0
    for X_b, _ in train_loader:
        x_hat, z = model_ae(X_b)
        loss = criterion(x_hat, X_b)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X_b.size(0)
    losses.append(epoch_loss / len(X_train))

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Training Loss")
plt.grid(True, alpha=0.3)
plt.show()
print(f"最终重建损失: {losses[-1]:.6f}")

In [ ]:
# 可视化潜在空间
model_ae.eval()
with torch.no_grad():
    _, z_test = model_ae(torch.from_numpy(X_test))
    z_test = z_test.numpy()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(z_test[:, 0], z_test[:, 1], z_test[:, 2], 
                     c=y_test, cmap="tab10", alpha=0.6, s=10)
ax.set_xlabel("z1")
ax.set_ylabel("z2")
ax.set_zlabel("z3")
ax.set_title("Autoencoder Latent Space (3D)")
plt.colorbar(scatter, label="Class")
plt.tight_layout()
plt.show()

print("20维数据被压缩到3维, 5个类别在潜在空间中自然分开")
print("这说明自编码器学到了数据的关键结构")

In [ ]:
# 重建质量分析
model_ae.eval()
with torch.no_grad():
    X_test_t = torch.from_numpy(X_test)
    X_hat, _ = model_ae(X_test_t)
    recon_error = ((X_test_t - X_hat)**2).mean(dim=1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(recon_error, bins=30, color="steelblue", alpha=0.7)
axes[0].set_xlabel("Reconstruction Error (MSE)")
axes[0].set_ylabel("Count")
axes[0].set_title("Reconstruction Error Distribution")
axes[0].axvline(np.mean(recon_error), color="red", linestyle="--", label=f"Mean={np.mean(recon_error):.4f}")
axes[0].legend()

# 原始 vs 重建
idx = 0
axes[1].bar(range(20), X_test[idx], alpha=0.5, label="Original")
axes[1].bar(range(20), X_hat[idx].numpy(), alpha=0.5, label="Reconstructed")
axes[1].set_xlabel("Feature Index")
axes[1].set_ylabel("Value")
axes[1].set_title("Original vs Reconstructed (Sample 0)")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 2. 去噪自编码器 (Denoising Autoencoder, DAE)

### 思路

输入加噪声, 输出还原无噪声的原始数据:

```
x_noisy = x + noise
x_noisy -> 编码器 -> z -> 解码器 -> x_hat

目标: x_hat 接近 x (无噪声的原始数据)
```

效果: 学到鲁棒的特征表示, 去除噪声干扰。

RFFI中特别有用: IQ信号常有噪声, DAE可以学到噪声不变的指纹特征。

In [ ]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim=20, latent_dim=3, noise_factor=0.3):
        super().__init__()
        self.noise_factor = noise_factor
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, input_dim),
        )
    
    def add_noise(self, x):
        noise = torch.randn_like(x) * self.noise_factor
        return x + noise
    
    def forward(self, x, add_noise=True):
        if add_noise and self.training:
            x_noisy = self.add_noise(x)
        else:
            x_noisy = x
        z = self.encoder(x_noisy)
        x_hat = self.decoder(z)
        return x_hat, z

model_dae = DenoisingAutoencoder(input_dim=20, latent_dim=3, noise_factor=0.3)
optimizer = optim.Adam(model_dae.parameters(), lr=1e-3)

losses_dae = []
for epoch in range(100):
    epoch_loss = 0
    for X_b, _ in train_loader:
        x_hat, z = model_dae(X_b, add_noise=True)
        loss = nn.MSELoss()(x_hat, X_b)  # 目标是干净数据!
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X_b.size(0)
    losses_dae.append(epoch_loss / len(X_train))

print(f"DAE 最终重建损失: {losses_dae[-1]:.6f}")

In [ ]:
# 去噪效果可视化
model_dae.eval()
X_test_t = torch.from_numpy(X_test)
noise = torch.randn_like(X_test_t) * 0.3
X_noisy = X_test_t + noise

with torch.no_grad():
    X_denoised, _ = model_dae(X_noisy, add_noise=False)

idx = 0
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(range(20), X_test[idx], color="steelblue")
axes[0].set_title("Original (Clean)")
axes[0].set_ylim(-3, 3)

axes[1].bar(range(20), X_noisy[idx].numpy(), color="coral")
axes[1].set_title("Noisy Input")
axes[1].set_ylim(-3, 3)

axes[2].bar(range(20), X_denoised[idx].numpy(), color="seagreen")
axes[2].set_title("Denoised Output")
axes[2].set_ylim(-3, 3)

plt.suptitle("Denoising Autoencoder Effect", fontsize=13)
plt.tight_layout()
plt.show()

mse_noisy = ((X_test_t - X_noisy)**2).mean().item()
mse_denoised = ((X_test_t - X_denoised)**2).mean().item()
print(f"噪声输入 MSE: {mse_noisy:.4f}")
print(f"去噪输出 MSE: {mse_denoised:.4f}")
print(f"噪声降低: {(1 - mse_denoised/mse_noisy)*100:.1f}%")

---
## 3. 变分自编码器 (Variational Autoencoder, VAE)

### 与普通自编码器的区别

```
普通AE: 编码器输出 z (确定值)
VAE:    编码器输出 mu, log_var (分布参数)
        z = mu + std * epsilon, epsilon ~ N(0,1)
```

VAE的潜在空间是连续的、结构化的, 可以生成新样本。

### VAE损失

```
L = 重建损失 + KL散度
  = MSE(x, x_hat) + KL(q(z|x) || p(z))

KL散度: 让编码分布 q(z|x) 接近标准正态分布 N(0,1)
       使潜在空间连续, 便于生成
```

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=20, latent_dim=3, hidden_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )
    
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z

def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    recon_loss = F.mse_loss(x_hat, x, reduction="sum")
    kl_loss = -0.5 * torch.sum(1 + logvar - mu**2 - torch.exp(logvar))
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

model_vae = VAE(input_dim=20, latent_dim=3)
print(f"VAE参数量: {sum(p.numel() for p in model_vae.parameters()):,}")

In [ ]:
# 训练VAE
optimizer = optim.Adam(model_vae.parameters(), lr=1e-3)
total_losses, recon_losses, kl_losses = [], [], []

for epoch in range(100):
    t_loss, r_loss, k_loss = 0, 0, 0
    for X_b, _ in train_loader:
        x_hat, mu, logvar, z = model_vae(X_b)
        loss, recon, kl = vae_loss(X_b, x_hat, mu, logvar)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
        r_loss += recon.item()
        k_loss += kl.item()
    total_losses.append(t_loss / len(X_train))
    recon_losses.append(r_loss / len(X_train))
    kl_losses.append(k_loss / len(X_train))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(total_losses)
axes[0].set_title("Total Loss")
axes[1].plot(recon_losses)
axes[1].set_title("Reconstruction Loss")
axes[2].plot(kl_losses)
axes[2].set_title("KL Divergence")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# VAE潜在空间可视化
model_vae.eval()
with torch.no_grad():
    _, mu_test, _, z_test = model_vae(torch.from_numpy(X_test))

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(mu_test[:, 0].numpy(), mu_test[:, 1].numpy(), mu_test[:, 2].numpy(),
                     c=y_test, cmap="tab10", alpha=0.6, s=10)
ax.set_xlabel("mu1")
ax.set_ylabel("mu2")
ax.set_zlabel("mu3")
ax.set_title("VAE Latent Space (mu)")
plt.colorbar(scatter, label="Class")
plt.tight_layout()
plt.show()

print("VAE的潜在空间更连续、更规则 (受KL约束接近标准正态)")
print("可以在潜在空间中插值生成新样本")

In [ ]:
# VAE生成新样本
model_vae.eval()
with torch.no_grad():
    z_sample = torch.randn(10, 3)  # 从标准正态采样
    generated = model_vae.decode(z_sample)

fig, axes = plt.subplots(2, 5, figsize=(15, 5))
for i, ax in enumerate(axes.flatten()):
    ax.bar(range(20), generated[i].numpy(), color="steelblue", alpha=0.7)
    ax.set_ylim(-3, 3)
    ax.set_title(f"Sample {i+1}")
plt.suptitle("VAE Generated Samples", fontsize=13)
plt.tight_layout()
plt.show()
print("从标准正态分布采样z, 解码器生成新样本")
print("这是普通AE做不到的, 因为AE的潜在空间不连续")

In [ ]:
# AE vs VAE 潜在空间对比
print("=== AE vs VAE 对比 ===")
print()
print("AE (普通自编码器):")
print("  潜在空间: 不连续, 有空洞")
print("  生成能力: 差, 随机采样可能生成无意义样本")
print("  用途: 特征提取, 降维, 去噪")
print()
print("VAE (变分自编码器):")
print("  潜在空间: 连续, 结构化 (受KL约束)")
print("  生成能力: 好, 可以从潜在空间采样生成")
print("  用途: 生成模型, 数据增强, 特征提取")
print()
print("重参数化技巧 (Reparameterization Trick):")
print("  z = mu + std * epsilon, epsilon ~ N(0,1)")
print("  让梯度可以流过采样操作")
print()
print("beta-VAE:")
print("  L = recon_loss + beta * KL_loss")
print("  beta > 1: 更强调潜在空间的规则性 (更解耦)")
print("  beta < 1: 更强调重建质量")

---
## 4. 稀疏自编码器

### 思路

不限制瓶颈层维度, 而是限制激活的稀疏性:

```
L = MSE(x, x_hat) + lambda * sparsity_penalty

sparsity_penalty:
  L1正则: lambda * ||z||_1
  KL稀疏: lambda * KL(rho || rho_hat)
    rho: 目标激活率 (如0.05)
    rho_hat: 实际平均激活率
```

效果: 即使潜在维度 > 输入维度, 也能学到有意义的特征 (过完备表示)。

In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim=20, latent_dim=40, sparsity_weight=1e-3):
        super().__init__()
        self.sparsity_weight = sparsity_weight
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, latent_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, input_dim),
        )
    
    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        sparsity_loss = torch.mean(torch.abs(z))  # L1稀疏惩罚
        return x_hat, z, sparsity_loss

model_sparse = SparseAutoencoder(input_dim=20, latent_dim=40, sparsity_weight=1e-3)
optimizer = optim.Adam(model_sparse.parameters(), lr=1e-3)

for epoch in range(100):
    for X_b, _ in train_loader:
        x_hat, z, sp_loss = model_sparse(X_b)
        recon_loss = F.mse_loss(x_hat, X_b)
        loss = recon_loss + model_sparse.sparsity_weight * sp_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# 查看稀疏性
model_sparse.eval()
with torch.no_grad():
    _, z_test, _ = model_sparse(torch.from_numpy(X_test))
    activation_rate = (z_test > 0.1).float().mean().item()

print(f"潜在维度: 40 (大于输入维度20, 过完备)")
print(f"激活率: {activation_rate:.4f} (大部分神经元不激活)")
print(f"稀疏性: 每个样本只激活少量神经元, 不同样本激活不同的神经元")

In [ ]:
# 稀疏自编码器激活模式
model_sparse.eval()
with torch.no_grad():
    _, z_vis, _ = model_sparse(torch.from_numpy(X_test[:50]))

plt.figure(figsize=(14, 6))
plt.imshow(z_vis.numpy() > 0.1, cmap="Blues", aspect="auto")
plt.xlabel("Latent Dimension")
plt.ylabel("Sample Index")
plt.title("Sparse Autoencoder Activation Pattern (blue=active)")
plt.colorbar(label="Active")
plt.tight_layout()
plt.show()
print("每个样本只激活少量神经元 (蓝色), 形成稀疏表示")
print("不同类别的样本激活不同的神经元组合")

---
## 5. 自编码器在RFFI中的应用

自编码器在射频指纹识别中有多种重要应用。

In [ ]:
# 应用1: 异常检测 (Open-set认证)
print("=== 应用1: 异常检测 (Open-set认证) ===")
print()
print("思路:")
print("  1. 只用合法设备的IQ数据训练AE")
print("  2. AE学会重建合法信号")
print("  3. 未知设备的信号重建误差大 -> 判定为异常")
print()
print("优势:")
print("  不需要未知设备的数据 (无监督)")
print("  天然支持Open-set")
print("  新设备出现时无需重新训练")

In [ ]:
# 异常检测实验
# 只用类别0-3训练, 类别4作为异常
normal_mask = y_train < 4
X_normal = X_train[normal_mask]
normal_ds = TensorDataset(torch.from_numpy(X_normal), torch.from_numpy(y_train[normal_mask]))
normal_loader = DataLoader(normal_ds, batch_size=64, shuffle=True)

anomaly_ae = Autoencoder(input_dim=20, latent_dim=3)
optimizer = optim.Adam(anomaly_ae.parameters(), lr=1e-3)

for epoch in range(100):
    for X_b, _ in normal_loader:
        x_hat, z = anomaly_ae(X_b)
        loss = F.mse_loss(x_hat, X_b)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# 测试
anomaly_ae.eval()
with torch.no_grad():
    X_hat_test, _ = anomaly_ae(torch.from_numpy(X_test))
    recon_errors = ((torch.from_numpy(X_test) - X_hat_test)**2).mean(dim=1).numpy()

is_anomaly = y_test >= 4  # 类别4为异常

plt.figure(figsize=(10, 5))
plt.hist(recon_errors[~is_anomaly], bins=30, alpha=0.6, label="Normal (Class 0-3)", color="blue", density=True)
plt.hist(recon_errors[is_anomaly], bins=30, alpha=0.6, label="Anomaly (Class 4)", color="red", density=True)
plt.xlabel("Reconstruction Error")
plt.ylabel("Density")
plt.title("Anomaly Detection via Autoencoder")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# ROC
from sklearn.metrics import roc_auc_score
auc_score = roc_auc_score(is_anomaly.astype(int), recon_errors)
print(f"异常检测 AUROC: {auc_score:.4f}")
print("异常样本重建误差更大, 可以用阈值区分")

In [ ]:
# 应用2: 特征提取 (预训练)
print("=== 应用2: 特征提取 (预训练) ===")
print()
print("思路:")
print("  1. 用大量无标签IQ数据训练AE")
print("  2. 去掉解码器, 用编码器作为特征提取器")
print("  3. 在编码器后接分类器, 用少量标签数据微调")
print()
print("优势:")
print("  利用无标签数据 (半监督学习)")
print("  标签数据少时效果显著")
print()
print("RFFI场景:")
print("  IQ数据容易大量采集, 但标注(设备ID)成本高")
print("  AE预训练 + 少量标签微调 = 实用方案")

In [ ]:
# 应用3: 信号去噪
print("=== 应用3: 信号去噪 ===")
print()
print("DAE可以直接用于IQ信号去噪:")
print("  输入: 含噪IQ信号")
print("  输出: 去噪IQ信号")
print()
print("训练策略:")
print("  1. 采集干净信号 (高SNR环境)")
print("  2. 人工加噪声生成训练对")
print("  3. 训练DAE: noisy -> clean")
print()
print("应用:")
print("  低SNR环境下的信号预处理")
print("  提高后续指纹识别的准确率")

In [ ]:
# 应用4: 数据增强 (VAE生成)
print("=== 应用4: 数据增强 (VAE生成) ===")
print()
print("VAE可以生成新的IQ样本:")
print("  1. 用现有数据训练VAE")
print("  2. 在潜在空间插值/采样")
print("  3. 解码生成新样本")
print()
print("优势:")
print("  生成样本保持设备指纹特征")
print("  可以扩充训练集, 尤其是少数类")
print()
print("注意事项:")
print("  生成质量需要验证")
print("  VAE生成的样本可能过于平滑")
print("  可以结合GAN提高生成质量")

In [ ]:
# 应用5: 域适应
print("=== 应用5: 域适应 ===")
print()
print("用AE对齐源域和目标域的特征分布:")
print()
print("方法1: 域对抗自编码器")
print("  编码器: 提取域不变特征")
print("  域判别器: 试图区分源域/目标域")
print("  对抗训练: 让编码器骗过域判别器")
print()
print("方法2: 对抗自编码器 (AAE)")
print("  编码器: x -> z")
print("  解码器: z -> x_hat")
print("  判别器: 判断z来自真实分布还是编码分布")
print("  让编码分布匹配目标分布")
print()
print("RFFI场景:")
print("  源域: 实验室高SNR数据")
print("  目标域: 实际部署低SNR数据")
print("  AE域适应: 让特征在不同SNR下一致")

---
## 6. 对比学习简述

对比学习是自监督学习的另一种范式, 与自编码器互补。

```
自编码器: 学习重建输入 (生成式)
对比学习: 学习区分不同样本 (判别式)
```

核心思想: 相似的样本在嵌入空间中靠近, 不相似的远离。

In [ ]:
# SimCLR 简化实现
class SimCLR(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=64, proj_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.projector = nn.Sequential(
            nn.Linear(hidden_dim, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim),
        )
    
    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, z

def nt_xent_loss(z1, z2, temperature=0.5):
    """NT-Xent (Normalized Temperature-scaled Cross Entropy)"""
    batch_size = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)
    z = F.normalize(z, dim=1)
    sim = z @ z.T / temperature
    
    labels = torch.cat([torch.arange(batch_size, 2*batch_size),
                        torch.arange(0, batch_size)])
    
    mask = torch.eye(2*batch_size, dtype=torch.bool)
    sim.masked_fill_(mask, -1e9)
    
    loss = F.cross_entropy(sim, labels)
    return loss

# 数据增强: 简单加噪声
def augment(x, noise_std=0.1):
    return x + torch.randn_like(x) * noise_std

# 训练
simclr = SimCLR(input_dim=20)
optimizer = optim.Adam(simclr.parameters(), lr=1e-3)

losses_simclr = []
for epoch in range(50):
    epoch_loss = 0
    for X_b, _ in train_loader:
        x1 = augment(X_b)
        x2 = augment(X_b)
        _, z1 = simclr(x1)
        _, z2 = simclr(x2)
        loss = nt_xent_loss(z1, z2)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses_simclr.append(epoch_loss / len(train_loader))

print(f"SimCLR训练完成, 最终损失: {losses_simclr[-1]:.4f}")

In [ ]:
# 对比学习 vs 自编码器
print("=== 对比学习 vs 自编码器 ===")
print()
print(f"{'维度':<15} {'自编码器':<25} {'对比学习':<25}")
print("-" * 65)
print(f"{'学习目标':<15} {'重建输入':<25} {'区分正负样本对':<25}")
print(f"{'损失函数':<15} {'MSE + KL':<25} {'NT-Xent / InfoNCE':<25}")
print(f"{'生成能力':<15} {'有(VAE)':<25} {'无':<25}")
print(f"{'特征质量':<15} {'中等':<25} {'通常更好':<25}")
print(f"{'数据增强':<15} {'不需要':<25} {'必须(构造正对)':<25}")
print(f"{'RFFI去噪':<15} {'DAE直接适用':<25} {'间接':<25}")
print(f"{'RFFI异常检测':<15} {'重建误差法':<25} {'嵌入距离法':<25}")
print()
print("RFFI推荐组合:")
print("  DAE去噪 + 对比学习特征 + ArcFace分类")
print("  或: VAE数据增强 + 有监督微调")

---
## 总结

### 自编码器家族

| 类型 | 核心改进 | 损失函数 | 主要用途 |
|------|---------|---------|----------|
| AE | 瓶颈压缩 | MSE | 降维, 特征提取 |
| DAE | 输入加噪 | MSE(去噪) | 去噪, 鲁棒特征 |
| VAE | 概率编码 | MSE + KL | 生成, 数据增强 |
| Sparse AE | 稀疏约束 | MSE + L1/KL | 过完备表示, 特征选择 |
| AAE | 对抗训练 | MSE + GAN | 域适应, 生成 |

### RFFI中的应用

| 应用 | 使用哪种AE | 原理 |
|------|-----------|------|
| 异常检测 | AE/VAE | 未知设备重建误差大 |
| 特征提取 | AE | 编码器作为预训练特征提取器 |
| 信号去噪 | DAE | 训练时加噪, 推理时去噪 |
| 数据增强 | VAE | 潜在空间采样生成新样本 |
| 域适应 | AAE | 对抗训练对齐特征分布 |

### 关键公式

```
AE:    L = ||x - D(E(x))||^2
DAE:   L = ||x - D(E(x + noise))||^2
VAE:   L = MSE(x, x_hat) + KL(q(z|x) || p(z))
       z = mu + std * epsilon  (重参数化)
Sparse: L = MSE + lambda * ||z||_1
```